# Drug-Target Binding Affinity Prediction using GNN
## EGN6217 — Engineering Applications of Machine Learning
### Sathyadharini Srinivasan | Spring 2026 | University of Florida

**Goal:** Predict binding affinity (Kd in nM) between a drug molecule and a protein target using a Graph Neural Network (GNN) + 1D CNN dual-branch architecture.

**Dataset:** DeepDTA Davis Dataset — 442 drugs × 68 proteins = 30,056 drug-target pairs.

---

## Cell 1 — Install Dependencies

In [ ]:
# Install required libraries
!pip install torch-geometric rdkit-pypi lifelines seaborn --quiet
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html --quiet

print("✓ All dependencies installed successfully")

## Cell 2 — Import Libraries

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from collections import Counter

# Set style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print(f"✓ PyTorch version:  {torch.__version__}")
print(f"✓ GPU available:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")
print(f"✓ NumPy version:    {np.__version__}")
print(f"✓ Pandas version:   {pd.__version__}")

## Cell 3 — Download Davis Dataset

In [ ]:
import os

os.makedirs('data/davis', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Download Davis dataset from DeepDTA GitHub
print("Downloading Davis dataset...")
!wget -q https://github.com/hkmztrk/DeepDTA/raw/master/data/davis/ligands_can.txt -O data/davis/ligands_can.txt
!wget -q https://github.com/hkmztrk/DeepDTA/raw/master/data/davis/proteins.txt   -O data/davis/proteins.txt
!wget -q https://github.com/hkmztrk/DeepDTA/raw/master/data/davis/Y             -O data/davis/Y

# Download train/test fold splits
!mkdir -p data/davis/folds
!wget -q https://github.com/hkmztrk/DeepDTA/raw/master/data/davis/folds/train_fold_setting1.txt -O data/davis/folds/train_fold_setting1.txt
!wget -q https://github.com/hkmztrk/DeepDTA/raw/master/data/davis/folds/test_fold_setting1.txt  -O data/davis/folds/test_fold_setting1.txt

print("✓ Dataset downloaded successfully")
print("Files in data/davis/:")
for f in os.listdir('data/davis'):
    size = os.path.getsize(f'data/davis/{f}') if os.path.isfile(f'data/davis/{f}') else '-'
    print(f"  {f}  ({size} bytes)" if isinstance(size, int) else f"  {f}/")

## Cell 4 — Load Dataset

In [ ]:
# Load drug SMILES
with open('data/davis/ligands_can.txt') as f:
    ligands = json.load(f)
smiles_list = list(ligands.values())
drug_names  = list(ligands.keys())

# Load protein sequences
with open('data/davis/proteins.txt') as f:
    proteins = json.load(f)
protein_seqs  = list(proteins.values())
protein_names = list(proteins.keys())

# Load affinity matrix (proteins x drugs)
affinity = pd.read_csv('data/davis/Y', header=None, sep='\t')

print("=" * 50)
print("DAVIS DATASET — SUMMARY")
print("=" * 50)
print(f"  Drug compounds:        {len(smiles_list):>6}")
print(f"  Protein targets:       {len(protein_seqs):>6}")
print(f"  Drug-target pairs:     {len(smiles_list)*len(protein_seqs):>6}")
print(f"  Affinity matrix shape: {affinity.shape}")
print()
print("Affinity (Kd) statistics:")
aff_vals = affinity.values.flatten()
print(f"  Min:    {aff_vals.min():.2f} nM")
print(f"  Max:    {aff_vals.max():.2f} nM")
print(f"  Mean:   {aff_vals.mean():.2f} nM")
print(f"  Median: {np.median(aff_vals):.2f} nM")
print(f"  Std:    {aff_vals.std():.2f} nM")
print()
print("Example drug SMILES:")
print(f"  {drug_names[0]}: {smiles_list[0][:60]}...")
print()
print("Example protein sequence (first 60 chars):")
print(f"  {protein_names[0]}: {protein_seqs[0][:60]}...")

## Cell 5 — Validate SMILES with RDKit

In [ ]:
# Validate all SMILES strings
valid_smiles   = []
invalid_smiles = []
atom_counts    = []
bond_counts    = []
mol_weights    = []

for smi in smiles_list:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        valid_smiles.append(smi)
        atom_counts.append(mol.GetNumAtoms())
        bond_counts.append(mol.GetNumBonds())
        mol_weights.append(Descriptors.MolWt(mol))
    else:
        invalid_smiles.append(smi)

print(f"✓ Valid SMILES:   {len(valid_smiles)}/{len(smiles_list)}")
print(f"✗ Invalid SMILES: {len(invalid_smiles)}/{len(smiles_list)}")
print()
print("Molecular properties (valid drugs only):")
print(f"  Avg atoms per molecule: {np.mean(atom_counts):.1f}")
print(f"  Avg bonds per molecule: {np.mean(bond_counts):.1f}")
print(f"  Avg molecular weight:   {np.mean(mol_weights):.1f} Da")
print(f"  MW range: {min(mol_weights):.0f} – {max(mol_weights):.0f} Da")

## Cell 6 — Exploratory Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Davis Dataset — Exploratory Data Analysis', fontsize=14, fontweight='bold', y=1.01)

# Plot 1: Binding affinity distribution
ax = axes[0, 0]
ax.hist(aff_vals, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Binding Affinity Kd (nM)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Binding Affinities')
ax.axvline(np.median(aff_vals), color='red', linestyle='--', linewidth=1.2, label=f'Median={np.median(aff_vals):.0f}')
ax.legend(fontsize=9)

# Plot 2: Atom count distribution
ax = axes[0, 1]
ax.hist(atom_counts, bins=30, color='mediumseagreen', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Number of Atoms')
ax.set_ylabel('Count')
ax.set_title('Drug Molecule Size (Atom Count)')
ax.axvline(np.mean(atom_counts), color='red', linestyle='--', linewidth=1.2, label=f'Mean={np.mean(atom_counts):.1f}')
ax.legend(fontsize=9)

# Plot 3: Molecular weight distribution
ax = axes[1, 0]
ax.hist(mol_weights, bins=30, color='mediumpurple', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Molecular Weight (Da)')
ax.set_ylabel('Count')
ax.set_title('Drug Molecular Weight Distribution')
ax.axvline(500, color='orange', linestyle='--', linewidth=1.2, label='Lipinski MW=500')
ax.legend(fontsize=9)

# Plot 4: Protein sequence length distribution
seq_lengths = [len(s) for s in protein_seqs]
ax = axes[1, 1]
ax.hist(seq_lengths, bins=20, color='coral', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Sequence Length (amino acids)')
ax.set_ylabel('Count')
ax.set_title('Protein Sequence Length Distribution')
ax.axvline(1000, color='red', linestyle='--', linewidth=1.2, label='Max len=1000')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ EDA plots saved to results/eda_plots.png")

## Cell 7 — Molecule → Graph Conversion Demo

In [ ]:
import sys
sys.path.append('src')
from graph_utils import smiles_to_graph, encode_protein

# Test on first 5 drugs
print("Molecular Graph Conversion Test")
print("=" * 50)
success = 0
for i, (name, smi) in enumerate(list(ligands.items())[:5]):
    g = smiles_to_graph(smi)
    if g is not None:
        success += 1
        print(f"Drug {i+1}: {name}")
        print(f"  SMILES: {smi[:50]}...")
        print(f"  Nodes (atoms): {g.x.shape[0]}")
        print(f"  Edges (bonds): {g.edge_index.shape[1]//2}")
        print(f"  Node feature dim: {g.x.shape[1]}")
        print()

print(f"✓ Successfully converted {success}/5 test molecules to graphs")

# Test protein encoding
print("\nProtein Sequence Encoding Test")
print("=" * 50)
for i, (name, seq) in enumerate(list(proteins.items())[:2]):
    enc = encode_protein(seq)
    print(f"Protein: {name}")
    print(f"  Sequence length: {len(seq)} amino acids")
    print(f"  Encoded tensor shape: {enc.shape}")
    print(f"  Non-zero positions: {enc.nonzero().shape[0]}")
    print()

## Cell 8 — Model Architecture Preview

In [ ]:
from model import DTAModel, count_parameters

model = DTAModel()
total_params = count_parameters(model)

print("DTAModel Architecture")
print("=" * 50)
print(model)
print()
print(f"Total trainable parameters: {total_params:,}")
print()
print("Input/Output dimensions:")
print("  Drug:    SMILES → molecular graph → GCN → 128-dim")
print("  Protein: AA sequence → Conv1D → 96-dim")
print("  Fusion:  concat(128+96=224) → MLP → 1 scalar")
print()
print("✓ Model instantiated and ready for training")

## Cell 9 — Environment Verification Summary

In [ ]:
print("=" * 55)
print(" ENVIRONMENT VERIFICATION — COMPLETE")
print("=" * 55)
print(f" ✓  Dataset loaded       — {len(smiles_list)} drugs, {len(protein_seqs)} proteins")
print(f" ✓  Valid SMILES         — {len(valid_smiles)}/{len(smiles_list)} molecules")
print(f" ✓  Graph conversion     — SMILES → PyG Data objects")
print(f" ✓  Protein encoding     — AA sequences → integer tensors")
print(f" ✓  EDA plots saved      — results/eda_plots.png")
print(f" ✓  Model architecture   — {total_params:,} parameters")
print(f" ✓  GPU status           — {'Available (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU only (training will be slower)'}")
print("=" * 55)
print(" Ready to proceed to Week 2: baseline training")
print("=" * 55)